In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_35.pickle

In [ ]:
%%PandasProfile
### cell 35 ###

import numpy as np

# restore original question strings
question_of_interest_old_cell47 = \
    'What machine learning frameworks have you used in the past 5 years?'
question_of_interest_new_cell47 = \
    'Which of the following machine learning frameworks do you use on a regular basis?'
question_of_interest_cell47 = question_of_interest_new_cell47


def grab_subset_of_data_47(original_df, question_of_interest):
    subset_cols = [c for c in original_df.columns if question_of_interest in c]
    new_names   = [c.split('-')[-1].lstrip() for c in subset_cols]
    return (
        original_df.loc[:, subset_cols]
                   .rename(columns=dict(zip(subset_cols, new_names)))
                   .dropna(how='all', subset=new_names)
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
    question_of_interest,
    include_2017=False
):
    df_map = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10,
    }
    if include_2017:
        df_map['2017'] = responses_df_2017
    dfs = [
        grab_subset_of_data_47(df_map[y], question_of_interest).assign(year=y)
        for y in sorted(df_map)
    ]
    df_combined = pd.concat(dfs)
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_47(df, df_counts):
    df_pct = df_counts.copy()
    total = df['year'].value_counts().reindex(df_pct['year']).values
    cols  = df_pct.columns.difference(['year'])
    df_pct[cols] = df_pct[cols].mul(100).div(total, axis=0)
    return df_pct

# rename the 2018 columns once
df_old = responses_df_2018_cell10.columns
responses_df_2018_cell10.columns = (
    df_old.str.replace(
        question_of_interest_old_cell47,
        question_of_interest_new_cell47,
        regex=False
    )
)

# combine and tag per year
ml_df_combined_cell47, ml_df_combined_counts_cell47 = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest_cell47
    )
)

# consolidate framework columns in one pass
ml_df2 = ml_df_combined_cell47.copy()
mapping = {
    'TensorFlow/Keras': (
        ['TensorFlow', 'TensorFlow ', 'Keras', 'Keras '],
        'TensorFlow/Keras'
    ),
    'PyTorch/Lightning/Fast.ai': (
        ['PyTorch', 'PyTorch ', 'PyTorch Lightning ', 'Fast.ai ', 'Fastai'],
        'PyTorch/PyTorch Lightning/Fast.ai'
    ),
    'Xgboost/LightGBM/CatBoost': (
        ['Xgboost', 'Xgboost ', 'lightgbm', 'LightGBM ', 'catboost', 'CatBoost '],
        'Xgboost/LightGBM/CatBoost'
    ),
    'Scikit-learn': (
        ['Scikit—learn ', 'Scikit—learn ', 'learn ', 'Learn'],
        'Scikit-learn'
    )
}
drop_cols = []
for new_col, (cols, val) in mapping.items():
    mask = ml_df2[cols].notna().any(axis=1)
    ml_df2[new_col] = np.where(mask, val, np.nan)
    drop_cols.extend(cols)
ml_df2.drop(columns=drop_cols, inplace=True)

# regroup & convert to percentages
ml_df_counts2 = ml_df2.groupby('year').count().reset_index()
ml_df_pct     = convert_df_of_counts_to_percentages_47(ml_df2, ml_df_counts2)

# select & reshape for plotting
df_cell47 = (
    ml_df_pct.loc[:, [
        'year',
        'Scikit-learn',
        'TensorFlow/Keras',
        'PyTorch/Lightning/Fast.ai',
        'Xgboost/LightGBM/CatBoost'
    ]]
    .melt(
        id_vars=['year'],
        value_vars=[
            'Scikit-learn',
            'Xgboost/LightGBM/CatBoost',
            'TensorFlow/Keras',
            'PyTorch/Lightning/Fast.ai'
        ]
    )
    .sort_values(by=['year', 'value'], ascending=True)
    .rename(columns={'variable': ''})
)

df_cell47.info()